# DSA5206: improved Duong reproduction
This notebook preserves the original analysis in a separate file. Outputs go to a separate directory. No numerical results are supplied until execution with real data.

Run cells in order. On Colab choose a Python CPU runtime (GPU optional), clone this branch, install dependencies, then configure paths below. Upload/extract the matching historical chest X-ray images, `metadata_2020.csv`, and `covid-severity-scores.csv` to Drive or `/content`. Counts are checks, not proof of cohort identity. Do not substitute an arbitrary contemporary dataset.


In [ ]:
# Colab: run once. Restart the runtime if prompted, then continue below.
%pip install torch==2.5.1 torchvision==0.20.1 torchxrayvision==1.4.0 numpy==1.26.4 pandas==2.2.3 scipy==1.14.1 scikit-learn==1.5.2 scikit-image==0.24.0 matplotlib==3.9.2


In [ ]:
# Optional on Colab: mount Drive before configuring paths.
# from google.colab import drive
# drive.mount('/content/drive')
import os, sys, json, hashlib, platform, importlib.metadata
from pathlib import Path
# Clone improve-duong-reproduction first; adjust REPO_DIR if your checkout differs.
REPO_DIR = Path(os.environ.get('DUONG_REPO_DIR', '/content/nus_DSA5206_GROUPPROJECT')).resolve()
DATA_DIR = Path(os.environ.get('DUONG_DATA_DIR', '/content/drive/MyDrive/duong-data')).resolve()
IMAGE_DIR = Path(os.environ.get('DUONG_IMAGE_DIR', str(DATA_DIR / 'images')))
METADATA_CSV = Path(os.environ.get('DUONG_METADATA_CSV', str(DATA_DIR / 'metadata_2020.csv')))
SCORES_CSV = Path(os.environ.get('DUONG_SCORES_CSV', str(DATA_DIR / 'covid-severity-scores.csv')))
OUTPUT_DIR = Path(os.environ.get('DUONG_OUTPUT_DIR', '/content/duong-improved-results')).resolve()
assert OUTPUT_DIR != REPO_DIR and REPO_DIR not in OUTPUT_DIR.parents, 'Keep outputs outside the checkout'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_DIR))
os.chdir(OUTPUT_DIR)
TRIAL_SEEDS = list(range(50))
FIGURE2_SEED = 1
QUALITATIVE_SEEDS = (0,49)  # fixed comparisons; never tune for better metrics
SALIENCY_SEED = 0  # preserves original reproduction case selection
assert IMAGE_DIR.is_dir() and METADATA_CSV.is_file() and SCORES_CSV.is_file(), 'Supply the original data files first'


In [ ]:
#导入python库

import os,sys
sys.path.insert(0,"..")
sys.path.insert(0,"../..")
from glob import glob
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import sys
import torch.nn.functional as F
import pandas as pd
import torchxrayvision as xrv
from duong_methods import tolerance_accuracy, parameter_label, saliency_pair


In [ ]:
nn_model = xrv.models.DenseNet(weights="all", cache_dir=str(OUTPUT_DIR / 'weights'))
nn_model.op_threshs = None
nn_model.eval()
checkpoint = Path(nn_model.weights_filename_local)
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
provenance = {
    'python': platform.python_version(),
    'packages': {p: importlib.metadata.version(p) for p in ['torch','torchvision','torchxrayvision','numpy','pandas','scipy','scikit-learn','scikit-image','matplotlib']},
    'checkpoint': str(checkpoint), 'checkpoint_sha256': sha256_file(checkpoint),
    'weights_alias': 'all', 'model_pathologies': list(nn_model.pathologies),
    'metadata_sha256': sha256_file(METADATA_CSV), 'scores_sha256': sha256_file(SCORES_CSV),
    'trial_seeds': TRIAL_SEEDS, 'figure2_seed': FIGURE2_SEED,
    'qualitative_seeds': QUALITATIVE_SEEDS, 'saliency_seed': SALIENCY_SEED,
}
# Hashes record the files used; they do not establish identity with the 2020 study.
print(json.dumps(provenance, indent=2))
(OUTPUT_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2))


In [ ]:
#图像中心裁剪、缩放：224×224

import torchvision, torchvision.transforms
transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])

In [ ]:
dmerge = xrv.datasets.COVID19_Dataset(imgpath=str(IMAGE_DIR), csvpath=str(METADATA_CSV), transform=transform, views=['PA'])
scores = pd.read_csv(SCORES_CSV, skiprows=5).set_index('filename')
assert scores.index.is_unique
assert dmerge.csv.filename.is_unique
assert dmerge.csv.patientid.notna().all()
dmerge.csv = dmerge.csv.set_index('filename').join(scores, on='filename').reset_index()


In [ ]:
#匹配验证

print("Total images:", len(dmerge))
print("Scored images:", dmerge.csv["geographic_mean"].notna().sum())
print("Opacity labels:", dmerge.csv["opacity_mean"].notna().sum())
print("Unique scored patients:",
      dmerge.csv.loc[dmerge.csv["geographic_mean"].notna(), "patientid"].nunique())

In [ ]:
#补充的单张图片测试

nn_model.eval()

sample = dmerge[0]["img"]
print("Image shape:", sample.shape)

with torch.no_grad():
    x = torch.from_numpy(sample).unsqueeze(0).float()
    prediction = nn_model(x)

print("Prediction shape:", prediction.shape)
print("Prediction:", prediction)

In [ ]:
#检查缺失图片和评分情况

import os
import pandas as pd

# 查看全部208条记录对应的图片是否存在
dmerge.csv["file_exists"] = dmerge.csv["filename"].apply(
    lambda filename: os.path.isfile(
        os.path.join(dmerge.imgpath, filename)
    )
)

missing_images = dmerge.csv.loc[
    ~dmerge.csv["file_exists"],
    ["filename"]
]

print("Total records:", len(dmerge.csv))
print("Existing images:", dmerge.csv["file_exists"].sum())
print("Missing images:", len(missing_images))

print("\nMissing filenames:")
print(missing_images.to_string(index=False))

# 检查缺失图片是否有原论文评分
scored_col = "geographic_mean"

print("\nMissing images with severity scores:",
      dmerge.csv.loc[
          ~dmerge.csv["file_exists"],
          scored_col
      ].notna().sum())


In [ ]:
#提取94张胸片的1024维和18维特征

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# 1. 找到94张有评分的图像
scored_mask = (
    dmerge.csv["geographic_mean"].notna()
    & dmerge.csv["opacity_mean"].notna()
)

scored_indices = np.flatnonzero(scored_mask.to_numpy())

print("Scored images:", len(scored_indices))

# 2. 检查这些图像是否全部存在
for idx in scored_indices:
    filename = dmerge.csv.iloc[idx]["filename"]
    filepath = os.path.join(dmerge.imgpath, filename)
    assert os.path.isfile(filepath), f"Missing: {filename}"

assert len(scored_indices) == 94
print("All 94 scored images are available!")

# 3. 保存与特征一一对应的标签和患者信息
scored_metadata = (
    dmerge.csv.iloc[scored_indices].copy().reset_index(drop=True)
)

# 4. 只加载94张带评分图像
scored_dataset = torch.utils.data.Subset(
    dmerge, scored_indices.tolist()
)

scored_dl = torch.utils.data.DataLoader(
    scored_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# 5. 提取特征
nn_model.eval()

d_feats = []
d_preds = []

with torch.no_grad():
    for i, batch in enumerate(scored_dl):
        imgs = batch["img"]

        features = nn_model.features(imgs)
        features = F.relu(features, inplace=False)
        features = F.adaptive_avg_pool2d(features, (1, 1))

        predictions = nn_model(imgs)

        d_feats.append(features.cpu().numpy())
        d_preds.append(predictions.cpu().numpy())

        print(f"Batch {i+1}/{len(scored_dl)} completed")

# 6. 合并结果
d_feats = np.concatenate(d_feats).reshape(-1, 1024)
d_preds = np.concatenate(d_preds)

print("\nFeature extraction completed!")
print("d_feats shape:", d_feats.shape)
print("d_preds shape:", d_preds.shape)

# 7. 保存特征和对应标签
assert len(scored_metadata) == len(d_feats) == len(d_preds)

np.save("duong_d_feats_94.npy", d_feats)
np.save("duong_d_preds_94.npy", d_preds)
scored_metadata.to_csv(
    "duong_scored_metadata_94.csv",
    index=False
)

print("All files saved successfully!")



In [ ]:
#评估函数evaluate

import sklearn, sklearn.linear_model, sklearn.metrics, sklearn.model_selection
import scipy, scipy.stats
def evaluate(data, labels, title, plot=True, groups=None, seed=0, method="mse", target_str=""):
    X = data
    y = labels.astype(float)
    
    gss = sklearn.model_selection.GroupShuffleSplit(train_size=0.5,test_size=0.5, random_state=seed)
    train_inds, test_inds = next(gss.split(X, y, groups))
    assert set(np.asarray(groups)[train_inds]).isdisjoint(set(np.asarray(groups)[test_inds]))
    X_train, X_test, y_train, y_test = X[train_inds], X[test_inds], y.iloc[train_inds], y.iloc[test_inds]
    #print("X_train", X_train.shape, "X_test", X_test.shape)

    if method=="mse":
        model = sklearn.linear_model.LinearRegression()
    elif method=="huber":
        model = sklearn.linear_model.HuberRegressor()
    elif method=="mae":
        # loss = max(0, |y - p| - epsilon)
        model = sklearn.linear_model.SGDRegressor(loss="epsilon_insensitive", epsilon=0, l1_ratio=1, random_state=seed)
    
    model = model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    pdiff = y_test - y_pred
    
    res = {}
    res["name"] = title
    res["R^2"] = sklearn.metrics.r2_score(y_test, y_pred)
    res["Correlation"] = scipy.stats.pearsonr(y_pred,y_test)[0]
    res["< 0.7 ACC"] = tolerance_accuracy(y_test, y_pred)
    res["explained_variance"] = sklearn.metrics.explained_variance_score(y_test, y_pred)
    res["MAE"] = sklearn.metrics.mean_absolute_error(y_test, y_pred)
    res["MSE"] = sklearn.metrics.mean_squared_error(y_test, y_pred)
    res["method"] = method
    res["# params"] = parameter_label(X_train)

    
    if plot:
        fig, ax = plt.subplots(figsize=(6,4), dpi=100)
        for x,y,yp in zip(y_test,y_test,y_pred):
            plt.plot((x,x),(y,yp),color='red',marker='')

        pmax = int(np.max([y_pred.max(), y_test.max()]))+2
        plt.plot(range(pmax),range(pmax), c="gray", linestyle="--")
        plt.xlim(0,pmax-1)
        plt.ylim(0,pmax-1)

        plt.scatter(y_test, y_pred);
        plt.ylabel("Model prediction ($y_{pred}$)")
        plt.xlabel("Ground Truth ($y_{true}$)")
        plt.title(title);
        plt.text(0.01,0.97, "$R^2$={0:0.2f}".format(res["R^2"])+ "\n"+ 
                 "Correlation={0:0.2f}".format(res["Correlation"]), ha='left', va='top', transform=ax.transAxes)

    return res#, pdiff


## Quantitative evaluation
Use 94 scored PA images, 1024 pooled post-ReLU features or 18 pre-sigmoid outputs, and patient-grouped 50/50 splits. Keep OLS and Pearson/R²/MAE/MSE definitions unchanged. Baseline Pearson correlation is undefined for constant predictions and remains N/A; the paper's approximate zero is a reporting convention, not the mathematical Pearson coefficient. The zero-column surrogate has only one estimable intercept (0+1).


In [ ]:
#Table 1 的代码
#2个评分任务×5种特征组合×50次随机划分=500次线性回归实验

import numpy as np
import pandas as pd
import torchxrayvision as xrv

# 确保特征、标签及患者编号一一对应
assert d_feats.shape == (94, 1024)
assert d_preds.shape == (94, 18)
assert len(scored_metadata) == 94

groups = scored_metadata["patientid"].reset_index(drop=True)

# 论文比较的5种特征组合
pathologies = xrv.datasets.default_pathologies

four_features = [
    "Lung Opacity",
    "Pneumonia",
    "Infiltration",
    "Consolidation"
]

feature_sets = {
    "Intermediate network features": d_feats,
    "No data": np.zeros((94, 1)),
    "18 outputs": d_preds,
    "4 outputs": d_preds[:, [
        pathologies.index(name) for name in four_features
    ]],
    "lung opacity output": d_preds[:, [
        pathologies.index("Lung Opacity")
    ]]
}

# 两种严重程度评分
targets = {
    "Geographic Extent": "geographic_mean",
    "Opacity": "opacity_mean"
}

results_list = []

for target_name, target_column in targets.items():

    labels = scored_metadata[target_column].astype(float).reset_index(drop=True)

    assert labels.notna().all()

    for feature_name, data in feature_sets.items():

        print(f"Running: {target_name} / {feature_name}")

        for seed in TRIAL_SEEDS:
            result = evaluate(
                data,
                labels,
                feature_name,
                plot=False,
                groups=groups,
                seed=seed,
                method="mse",
                target_str=target_name
            )

            result["Target"] = target_name
            result["Seed"] = seed
            results_list.append(result)

results = pd.DataFrame(results_list)

# 汇总50次实验的均值和标准差
summary = (
    results.groupby(["Target", "name"])[
        ["Correlation", "R^2", "MAE", "MSE"]
    ]
    .agg(["mean", "std"])
    .round(3)
)

display(summary)

# 保存实验结果
results.to_csv("duong_reproduction_500_trials.csv", index=False)
summary.to_csv("duong_reproduction_summary.csv")

print("All 500 regression fits completed!")


In [ ]:
#Table 1的对比表格

import pandas as pd
from IPython.display import display

# 1. 读取已经保存的500次实验结果
results_saved = pd.read_csv(
    "duong_reproduction_500_trials.csv"
)

# 2. 计算我们的均值和标准差
metrics = ["Correlation", "R^2", "MAE", "MSE"]

ours = (
    results_saved.groupby(["Target", "name"])[metrics]
    .agg(["mean", "std"])
)

# 3. 原论文Table 1中的数据
# 格式：(Target, name, Correlation, R2, MAE, MSE)
paper_data = [
    ("Geographic Extent", "lung opacity output",
     "0.80±0.05", "0.60±0.09", "1.14±0.11", "2.06±0.34"),
    ("Geographic Extent", "4 outputs",
     "0.79±0.05", "0.57±0.10", "1.19±0.11", "2.17±0.37"),
    ("Geographic Extent", "18 outputs",
     "0.76±0.08", "0.47±0.16", "1.32±0.17", "2.73±0.89"),
    ("Geographic Extent", "Intermediate network features",
     "0.74±0.08", "0.43±0.16", "1.36±0.13", "2.88±0.58"),
    ("Geographic Extent", "No data",
     "0.00±0.00", "-0.08±0.10", "2.00±0.17", "5.60±0.95"),

    ("Opacity", "lung opacity output",
     "0.78±0.04", "0.58±0.09", "0.78±0.05", "0.86±0.11"),
    ("Opacity", "4 outputs",
     "0.78±0.04", "0.58±0.09", "0.76±0.05", "0.87±0.12"),
    ("Opacity", "18 outputs",
     "0.73±0.09", "0.44±0.16", "0.86±0.11", "1.15±0.33"),
    ("Opacity", "Intermediate network features",
     "0.66±0.08", "0.25±0.21", "1.01±0.09", "1.54±0.28"),
    ("Opacity", "No data",
     "0.00±0.00", "-0.08±0.10", "1.24±0.10", "2.26±0.36")
]

paper_columns = [
    "Task", "Features", "Correlation",
    "R2", "MAE", "MSE"
]

paper = pd.DataFrame(paper_data, columns=paper_columns)

# 4. 生成与原论文并排的比较表
rows = []

for _, p in paper.iterrows():
    key = (p["Task"], p["Features"])

    row = {
        "Task": p["Task"],
        "Features": p["Features"]
    }

    for metric in metrics:
        paper_col = "R2" if metric == "R^2" else metric

        mean = ours.loc[key, (metric, "mean")]
        std = ours.loc[key, (metric, "std")]

        row[f"{paper_col} (Paper)"] = p[paper_col]

        row[f"{paper_col} (Ours)"] = (
            f"{mean:.3f}±{std:.3f}"
            if pd.notna(mean) and pd.notna(std)
            else "N/A"
        )

    rows.append(row)

comparison = pd.DataFrame(rows)


# 5. 按照原论文 Table 1 的顺序排列实验结果

task_order = [
    "Opacity",
    "Geographic Extent"
]

feature_order = [
    "lung opacity output",
    "4 outputs",
    "18 outputs",
    "Intermediate network features",
    "No data"
]

comparison["Task"] = pd.Categorical(
    comparison["Task"],
    categories=task_order,
    ordered=True
)

comparison["Features"] = pd.Categorical(
    comparison["Features"],
    categories=feature_order,
    ordered=True
)

comparison = (
    comparison
    .sort_values(["Task", "Features"])
    .reset_index(drop=True)
)

# 6. 恢复普通字符串列，便于保存和展示
comparison["Task"] = comparison["Task"].astype(str)
comparison["Features"] = comparison["Features"].astype(str)


# 7. 显示完整对比表
display(comparison)

# 8. 保存CSV和HTML
comparison.to_csv(
    "Duong_Table1_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison.to_html(
    "Duong_Table1_Comparison.html",
    index=False
)

print("Comparison table saved successfully!")


In [ ]:
#Table 1的对比表格导出为图片


import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# 使用已经排好顺序的 comparison 数据表
assert len(comparison) == 10

# 更接近原论文的英文特征名称
feature_labels = {
    "lung opacity output": '"lung opacity" output',
    "4 outputs": "4 outputs",
    "18 outputs": "18 outputs",
    "Intermediate network features": "Intermediate network\nfeatures",
    "No data": "No data"
}

# 两组任务各5行
display_df = comparison.copy()

display_df["Features"] = display_df["Features"].map(
    feature_labels
)

display_df["Task"] = (
    ["Opacity\nScore"] + [""] * 4
    + ["Geographic\nExtent"] + [""] * 4
)

# 保持两组任务顺序
columns = [
    "Task", "Features",
    "Correlation (Paper)", "Correlation (Ours)",
    "R2 (Paper)", "R2 (Ours)",
    "MAE (Paper)", "MAE (Ours)",
    "MSE (Paper)", "MSE (Ours)"
]

display_df = display_df[columns]

# 构建高分辨率图表
fig, ax = plt.subplots(figsize=(20, 7.7), dpi=160)
ax.axis("off")

fig.patch.set_facecolor("white")

# 标题
fig.text(
    0.5, 0.94,
    "TABLE 1 | QUANTITATIVE PERFORMANCE COMPARISON",
    ha="center", va="center",
    fontsize=18, fontweight="bold",
    color="#202020"
)

fig.text(
    0.5, 0.902,
    "Original Study vs. Our Reproduction",
    ha="center", va="center",
    fontsize=12, color="#555555"
)

headers = [
    "Task", "Features",
    "Pearson\nPaper", "Pearson\nOurs",
    "R²\nPaper", "R²\nOurs",
    "MAE\nPaper", "MAE\nOurs",
    "MSE\nPaper", "MSE\nOurs"
]

# 将表格限制在画布内
bbox = [0.025, 0.17, 0.95, 0.68]

table = ax.table(
    cellText=display_df.values,
    colLabels=headers,
    cellLoc="center",
    colLoc="center",
    bbox=bbox
)

table.auto_set_font_size(False)
table.set_fontsize(10)

# 列宽比例（合计1）
widths = [
    0.115, 0.175,
    0.08875, 0.08875,
    0.08875, 0.08875,
    0.08875, 0.08875,
    0.08875, 0.08875
]

n_rows = len(display_df) + 1
n_cols = len(headers)

for (r, c), cell in table.get_celld().items():

    cell.set_width(bbox[2] * widths[c])

    # 减少网格线，让表格更接近学术论文
    cell.set_linewidth(0)
    cell.PAD = 0.035

    if r == 0:
        cell.set_facecolor("#E7E7E7")
        cell.set_text_props(
            weight="bold",
            color="#222222",
            fontsize=10
        )
    else:
        cell.set_facecolor(
            "#FFFFFF" if r % 2 else "#F5F5F5"
        )

        if c == 1:
            cell.set_text_props(
                ha="left",
                fontsize=9.5
            )

        # 将首行的任务名称加粗
        if c == 0 and r in (1, 6):
            cell.set_text_props(
                weight="bold",
                fontsize=10
            )

        # Paper与Ours使用细微背景区别
        if c in (3, 5, 7, 9):
            cell.set_facecolor(
                "#F0F2F4" if r % 2 else "#E8EBEE"
            )

# 顶部、表头下方和底部横线
x0 = bbox[0]
x1 = bbox[0] + bbox[2]

y_bottom = bbox[1]
y_top = bbox[1] + bbox[3]

row_height = bbox[3] / n_rows

for y, lw in [
    (y_top, 1.5),
    (y_top - row_height, 1.1),
    (y_bottom, 1.5)
]:
    ax.plot(
        [x0, x1], [y, y],
        transform=ax.transAxes,
        color="#303030",
        linewidth=lw
    )

# Opacity与Geographic Extent之间的分隔线
separator_y = y_top - row_height * 6

ax.plot(
    [x0, x1],
    [separator_y, separator_y],
    transform=ax.transAxes,
    color="#444444",
    linewidth=1.6
)

# 脚注
fig.text(
    0.5, 0.11,
    "Values are mean ± standard deviation across 50 "
    "patient-grouped train/test splits.",
    ha="center",
    fontsize=10,
    color="#444444"
)

fig.text(
    0.5, 0.077,
    "Paper: Cohen et al. (2020). "
    "Ours: DSA5206 reproduction.",
    ha="center",
    fontsize=9,
    color="#666666"
)


# 保存最终版本的Table 1
plt.savefig(
    "Duong_Table1_Comparison.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.savefig(
    "Duong_Table1_Comparison.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Final Table 1 saved successfully!")


In [ ]:
#Figure 2 的代码

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

# 1. 读取之前保存的94张胸片数据
d_preds = np.load("duong_d_preds_94.npy")
metadata = pd.read_csv("duong_scored_metadata_94.csv")

assert d_preds.shape == (94, 18)
assert len(metadata) == 94

# 2. 选择论文最优方案使用的 Lung Opacity 特征
feature_idx = xrv.datasets.default_pathologies.index(
    "Lung Opacity"
)

X = d_preds[:, [feature_idx]]
groups = metadata["patientid"]

# 3. 按患者分组，50%训练、50%测试
seed = FIGURE2_SEED

splitter = GroupShuffleSplit(
    train_size=0.5,
    test_size=0.5,
    random_state=seed
)

train_idx, test_idx = next(
    splitter.split(X, groups=groups)
)

print("Training images:", len(train_idx))
print("Testing images:", len(test_idx))

# 4. 分别训练两种评分的线性回归
tasks = [
    ("geographic_mean", "Geographic Extent", 8),
    ("opacity_mean", "Opacity Score", 6)
]

fig, axes = plt.subplots(
    1, 2, figsize=(12, 5), dpi=150
)

prediction_records = []

for ax, (column, title, score_max) in zip(axes, tasks):

    y = metadata[column].astype(float).to_numpy()

    model = LinearRegression()
    model.fit(X[train_idx], y[train_idx])

    y_true = y[test_idx]
    y_pred = model.predict(X[test_idx])

    r2 = r2_score(y_true, y_pred)
    corr = pearsonr(y_true, y_pred)[0]

    # 红线：真实评分与预测评分之间的误差
    for actual, predicted in zip(y_true, y_pred):
        ax.plot(
            [actual, actual],
            [actual, predicted],
            color="red",
            alpha=0.45,
            linewidth=1
        )

    # 散点：每张测试胸片
    ax.scatter(y_true, y_pred, s=22)

    # 虚线：完美预测
    ax.plot(
        [0, score_max + 1],
        [0, score_max + 1],
        linestyle="--",
        color="gray"
    )

    ax.set_xlim(0, score_max + 1)
    ax.set_ylim(-1.5, score_max + 1)

    ax.set_xlabel("Ground Truth")
    ax.set_ylabel("Model Prediction")
    ax.set_title(title)

    ax.text(
        0.03, 0.97,
        f"R² = {r2:.2f}\nCorrelation = {corr:.2f}",
        transform=ax.transAxes,
        va="top"
    )

    # 保存每张测试图片的真实值和预测值
    for idx, actual, predicted in zip(
        test_idx, y_true, y_pred
    ):
        prediction_records.append({
            "Task": title,
            "Filename": metadata.iloc[idx]["filename"],
            "PatientID": metadata.iloc[idx]["patientid"],
            "Ground Truth": actual,
            "Prediction": predicted,
            "Error": predicted - actual
        })

    print(f"{title}: R²={r2:.3f}, Correlation={corr:.3f}")

# 5. 保存图片和预测数据
plt.tight_layout()

plt.savefig(
    "Duong_Figure2_Reproduction.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Duong_Figure2_Reproduction.pdf",
    bbox_inches="tight"
)

plt.show()

pd.DataFrame(prediction_records).to_csv(
    "Duong_Figure2_Predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Figure 2 reproduction saved successfully!")


In [ ]:
#额外114张胸片的特征提取，总共208张特征

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit

# 加载已经保存的94张胸片特征
preds_94 = np.load("duong_d_preds_94.npy")
meta_94 = pd.read_csv("duong_scored_metadata_94.csv")

# 使用原始数据表，保留所有实际存在的图片
meta_all = dmerge.csv.copy().reset_index(drop=True)

meta_all = meta_all[
    meta_all["filename"].map(
        lambda f: __import__("os").path.isfile(
            __import__("os").path.join(dmerge.imgpath, f)
        )
    )
].copy()

# 保留在原数据集中的行号
original_indices = meta_all.index.to_numpy()
meta_all = meta_all.reset_index(drop=True)

assert len(meta_all) == 208
assert meta_all["filename"].is_unique
assert meta_94["filename"].is_unique

# 根据文件名复用94张图片已经提取的特征
saved_features = dict(zip(meta_94["filename"], preds_94))

preds_all = np.zeros((len(meta_all), 18), dtype=np.float32)
missing_positions = []

for pos, filename in enumerate(meta_all["filename"]):
    if filename in saved_features:
        preds_all[pos] = saved_features[filename]
    else:
        missing_positions.append(pos)

print("Reused features:", len(meta_all) - len(missing_positions))
print("Images requiring extraction:", len(missing_positions))

# 只处理尚未提取的113张图片
extra_dataset = torch.utils.data.Subset(
    dmerge,
    original_indices[missing_positions].tolist()
)

extra_dl = torch.utils.data.DataLoader(
    extra_dataset, batch_size=8,
    shuffle=False, num_workers=0
)

nn_model.eval()
new_preds = []

with torch.no_grad():
    for i, batch in enumerate(extra_dl):
        pred = nn_model(batch["img"])
        new_preds.append(pred.cpu().numpy())
        print(f"Extra batch {i+1}/{len(extra_dl)} completed")

preds_all[missing_positions] = np.concatenate(new_preds)

assert preds_all.shape == (208, 18)

# 保存完整特征及行顺序
np.save("duong_d_preds_208.npy", preds_all)
meta_all.to_csv("duong_metadata_208.csv", index=False)

print("Full feature extraction completed!")
print("Feature shape:", preds_all.shape)


## Figure 3: fixed seed comparison
The author's cells 22–23 embed exactly four **pre-sigmoid** outputs: Lung Opacity, Pneumonia, Infiltration, Consolidation. This matches our representation. No standardization is added. Use all 208 available PA images, preserve filename order, and reuse the same embedding for both regressions.

Explicit t-SNE: 2 dimensions, perplexity 30, learning rate 200, random initialization, seed 54, Euclidean distance, Barnes–Hut, 1000 iterations (scikit-learn 1.5.2 defaults for unspecified parameters). The historical environment is not pinned, so identical coordinates are not guaranteed. The author notebook's saved embedding is (199,2), while paper Figure 3 states 208; neither a 199-image manifest nor a dataset revision explaining the nine images is included. Cause unresolved; do not remove images to force agreement.

Seed 0 preserves the existing comparison; seed 49 matches the author's final loop variable used for the qualitative regression. Both use identical features and t-SNE coordinates. Predictions include training images and are exploratory, not independent survival validation.


In [ ]:
#Figure 3 的代码（使用 scikit-learn 的 t-SNE）

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.manifold import TSNE

preds_all = np.load("duong_d_preds_208.npy")
meta_all = pd.read_csv("duong_metadata_208.csv")

names = [
    "Lung Opacity",
    "Pneumonia",
    "Infiltration",
    "Consolidation"
]

indices = [
    xrv.datasets.default_pathologies.index(name)
    for name in names
]

X4 = preds_all[:, indices]

print("scikit-learn version:", sklearn.__version__)
print("Feature shape:", X4.shape)

# 显式设置 t-SNE 的参数，便于记录和重现
embedding = TSNE(
    n_components=2,
    random_state=54,
    perplexity=30,
    learning_rate=200,
    init="random"
).fit_transform(X4)

# 使用原作者的生存状态分类
survival = meta_all["survival"].fillna("").astype(str)

categories = {
    "Unknown survival": ~survival.isin(["Y", "N"]),
    "Survived": survival == "Y",
    "Deceased": survival == "N"
}

assert len(meta_all) == len(preds_all) == 208
assert meta_all.filename.is_unique
assert set(meta_94.filename).issubset(set(meta_all.filename))
# Export exact ordered manifests used by the embedding and supervised model.
meta_all.to_csv('figure3_ordered_manifest.csv', index=False)
meta_94.to_csv('scored_ordered_manifest.csv', index=False)


In [ ]:
for qualitative_seed in QUALITATIVE_SEEDS:
    from sklearn.linear_model import LinearRegression
    from sklearn.model_selection import GroupShuffleSplit
    
    preds_94 = np.load("duong_d_preds_94.npy")
    meta_94 = pd.read_csv("duong_scored_metadata_94.csv")
    
    opacity_idx = xrv.datasets.default_pathologies.index("Lung Opacity")
    
    # 训练单特征线性回归模型
    X94 = preds_94[:, [opacity_idx]]
    y94 = meta_94["geographic_mean"].astype(float).to_numpy()
    
    splitter = GroupShuffleSplit(
        train_size=0.5,
        test_size=0.5,
        random_state=qualitative_seed
    )
    
    train_idx, test_idx = next(
        splitter.split(X94, y94, groups=meta_94["patientid"])
    )
    
    reg = LinearRegression()
    reg.fit(X94[train_idx], y94[train_idx])
    
    # 使用回归模型预测严重程度
    predicted_geo = reg.predict(
        preds_all[:, [opacity_idx]]
    )
    
    fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
    
    styles = {
        "Unknown survival": ("o", 5),
        "Survived": ("o", 65),
        "Deceased": ("x", 65)
    }
    
    for label, mask in categories.items():
        mask = mask.to_numpy()
        marker, size = styles[label]
    
        ax.scatter(
            embedding[mask, 0],
            embedding[mask, 1],
            c=predicted_geo[mask],
            s=size,
            marker=marker,
            cmap="winter",
            vmin=0,
            vmax=8,
            label=label
        )
    
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize
    
    colorbar = fig.colorbar(
        ScalarMappable(
            norm=Normalize(vmin=0, vmax=8),
            cmap="winter"
        ),
        ax=ax
    )
    
    colorbar.set_label("Geographic Extent Prediction")
    
    ax.set_title(
        "Figure 3: Model Predictions and Survival Outcomes"
    )
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend()
    
    plt.tight_layout()
    
    plt.savefig(
        f"Duong_Figure3_seed{qualitative_seed}.png",
        dpi=300,
        bbox_inches="tight"
    )
    
    plt.show()


## Figure 4: deliberately selected illustrative cases
Retain the seed-0 selection rules and images from the original reproduction: lowest-error severe/mild cases and extreme over/underestimation cases. These are post-selected illustrations, not a representative performance sample. With matching data, weights and versions the rules preserve the cases; source data are required to verify identity.

**A — Original Code Implementation:** squared score gradient; Gaussian sigma=(5,5), truncate=3.5, nearest-edge extension; effective 37×37 support. Matches released skimage code.

**B — Paper-Description Implementation (interpretation):** signed score gradient, convolved with a normalized discrete 5×5 Gaussian, sigma=1 pixel, reflect boundary (half-sample symmetric). The paper does not specify sigma, sign transformation or boundaries. B is an explicit interpretation, not an exact reconstruction.

Each pair uses identical input, checkpoint, fitted regression coefficients and gradient. A and B have different units (squared versus signed gradient), so sharing numerical limits is misleading. Use one global A scale across cases, and one symmetric global B scale across cases, identical overlay alpha and original-image limits. Colorbars expose each scale; panel intensities are not quantitatively comparable across methods.


In [ ]:
#Figure 4 的代码
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from skimage.filters import gaussian
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit

# ---------------------------
# 1. 重新准备测试集预测结果
# ---------------------------
preds_94 = np.load("duong_d_preds_94.npy")
meta_94 = pd.read_csv("duong_scored_metadata_94.csv")

opacity_idx = xrv.datasets.default_pathologies.index("Lung Opacity")

X = preds_94[:, [opacity_idx]]
y = meta_94["geographic_mean"].astype(float).to_numpy()
groups = meta_94["patientid"]

splitter = GroupShuffleSplit(
    train_size=0.5,
    test_size=0.5,
    random_state=SALIENCY_SEED
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

geo_model = LinearRegression()
geo_model.fit(X[train_idx], y[train_idx])

y_true = y[test_idx]
y_pred = geo_model.predict(X[test_idx])
errors = y_pred - y_true
abs_errors = np.abs(errors)

test_df = pd.DataFrame({
    "global_idx": test_idx,
    "filename": meta_94.iloc[test_idx]["filename"].values,
    "ground_truth": y_true,
    "prediction": y_pred,
    "error": errors,
    "abs_error": abs_errors
}).reset_index(drop=True)

print("Test set size:", len(test_df))

# ---------------------------
# 2. 自动选4个代表样本
# ---------------------------

# (a) 严重且预测较好：GT高，误差小
cand_a = test_df[test_df["ground_truth"] >= 4].sort_values(
    ["abs_error", "ground_truth"], ascending=[True, False]
).iloc[0]

# (b) 轻症且预测较好：GT低，误差小
cand_b = test_df[test_df["ground_truth"] <= 1].sort_values(
    ["abs_error", "ground_truth"], ascending=[True, True]
).iloc[0]

# (c) 明显高估：GT低，但预测偏高最多
cand_c = test_df[test_df["ground_truth"] <= 2].sort_values(
    "error", ascending=False
).iloc[0]

# (d) 明显低估：GT高，但预测偏低最多
cand_d = test_df[test_df["ground_truth"] >= 4].sort_values(
    "error", ascending=True
).iloc[0]

selected_cases = [
    ("(a) relatively correct / severe", cand_a),
    ("(b) relatively correct / mild", cand_b),
    ("(c) overestimation", cand_c),
    ("(d) underestimation", cand_d),
]

selected_df = pd.DataFrame([
    {
        "Case": name,
        "filename": row["filename"],
        "ground_truth": row["ground_truth"],
        "prediction": row["prediction"],
        "error": row["error"]
    }
    for name, row in selected_cases
])

display(selected_df)

# ---------------------------
# 3. 定义 saliency 计算函数
# ---------------------------
def make_saliency_for_idx(global_idx):
    sample = scored_dataset[int(global_idx)]
    img = sample["img"]

    x = torch.as_tensor(img, dtype=torch.float32).unsqueeze(0).clone().detach()
    x.requires_grad_(True)

    nn_model.eval()
    nn_model.zero_grad()

    outputs = nn_model(x)
    lung_opacity = outputs[:, opacity_idx]

    theta = float(geo_model.coef_[0])
    bias = float(geo_model.intercept_)

    predicted_extent = theta * lung_opacity + bias

    grads = torch.autograd.grad(predicted_extent.sum(), x)[0][0, 0].detach().cpu().numpy()

    a, b = saliency_pair(grads)
    return img[0], a, b, predicted_extent.item()



In [ ]:
panels = [make_saliency_for_idx(int(row['global_idx'])) for _, row in selected_cases]
a_max = max(float(a.max()) for _, a, b, pred in panels)
b_max = max(float(np.abs(b).max()) for _, a, b, pred in panels)
fig, axes = plt.subplots(4, 3, figsize=(15, 16))
for j, ((case, row), (img, a, b, pred)) in enumerate(zip(selected_cases, panels)):
    for ax in axes[j]:
        ax.imshow(img, cmap='gray', vmin=-1024, vmax=1024)
        ax.axis('off')
    axes[j,0].set_title(f"Deliberately selected: {case}\n{row['filename']}\nGT={row['ground_truth']:.1f}, Pred={pred:.2f}")
    im_a=axes[j,1].imshow(a, cmap='hot', alpha=.5, vmin=0, vmax=a_max)
    im_b=axes[j,2].imshow(b, cmap='coolwarm', alpha=.5, vmin=-b_max, vmax=b_max)
    axes[j,1].set_title('A: released code / squared gradient')
    axes[j,2].set_title('B: paper-description interpretation / signed gradient')
    fig.colorbar(im_a, ax=axes[j,1], fraction=.046)
    fig.colorbar(im_b, ax=axes[j,2], fraction=.046)
plt.tight_layout()
plt.savefig('Duong_Figure4_Methods_AB.png', dpi=300, bbox_inches='tight')
selected_df.to_csv('Duong_Figure4_SelectedCases.csv', index=False)
plt.show()
